# marketkit quickstart

A short tour of `marketkit`: fetching clean OHLCV data, computing risk/return
analytics, and a few common technical indicators.

Install the package first if you haven't:

```
pip install marketkit matplotlib
```

In [ ]:
import marketkit as mk

mk.__version__

## Fetch clean OHLCV data

`mk.get` returns a tidy `DataFrame` indexed by date, with adjusted prices and
predictable dtypes — no source-specific quirks to clean up.

In [ ]:
df = mk.get("AAPL", period="2y")
df.head()

In [ ]:
df.dtypes

## One-shot summary

`mk.summary` runs the common analytics for a ticker in one call.

In [ ]:
mk.summary("AAPL", period="2y")

## Returns and risk analytics

Each metric also works standalone, on the same `DataFrame`.

In [ ]:
daily_returns = mk.returns(df)
print("CAGR:       ", round(mk.cagr(df), 4))
print("Volatility: ", round(mk.volatility(df), 4))
print("Sharpe:     ", round(mk.sharpe(df), 2))
print("Sortino:    ", round(mk.sortino(df), 2))

dd_series, max_dd = mk.drawdown(df)
print("Max drawdown:", round(max_dd, 4))

In [ ]:
import matplotlib.pyplot as plt

dd_series.plot(figsize=(9, 3), title="AAPL drawdown")
plt.ylabel("drawdown")
plt.show()

## Technical indicators

Indicators take a `DataFrame` (or a `Series`) and return aligned `Series`/`DataFrame` output.

In [ ]:
df["sma50"] = mk.sma(df, window=50)
df["ema20"] = mk.ema(df, window=20)
df["rsi14"] = mk.rsi(df)

macd_df = mk.macd(df)
bb_df = mk.bollinger(df)

df[["close", "sma50", "ema20"]].tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

df[["close", "sma50", "ema20"]].plot(ax=axes[0], title="AAPL price + moving averages")
df["rsi14"].plot(ax=axes[1], title="RSI (14)")
axes[1].axhline(70, color="r", linestyle="--", linewidth=1)
axes[1].axhline(30, color="g", linestyle="--", linewidth=1)
macd_df[["macd", "signal"]].plot(ax=axes[2], title="MACD")

plt.tight_layout()
plt.show()

## Multiple tickers

Pass a list to fetch several tickers at once, either as a long (tidy) frame
or as a wide frame with one column level per ticker.

In [ ]:
long_df = mk.get(["AAPL", "MSFT", "GOOG"], period="1y")
long_df.head()

In [ ]:
wide_df = mk.get(["AAPL", "MSFT", "GOOG"], period="1y", wide=True)
wide_df["adj_close"].tail()

In [ ]:
wide_df["adj_close"].pct_change().add(1).cumprod().plot(
    figsize=(9, 4), title="Cumulative return"
)
plt.show()

## Caching and offline mode

Fetched data is cached locally as parquet (under a platform-appropriate cache
directory) and reused while still fresh. Pass `offline=True` to force
marketkit to only read from cache and never touch the network.

In [ ]:
cached_df = mk.get("AAPL", period="2y", offline=True)
cached_df.tail()